In [ ]:
from pathlib import Path
import sys
import os
import clingo
from typing import Any
import pandas as pd
project_root = "."


In [ ]:
from typing import Any
import clingo
import pandas as pd


State = dict[Any, Any]


def make_control(paths: list[str], limit: str | int) -> clingo.Control:
    ctl = clingo.Control([str(limit), "--project", "--warn=none"])
    for path in paths:
        ctl.load(path)
    return ctl


def states_from_reference_model(
    RMODEL: list[str],
    ASPGARPFILES: list[str],
) -> list[State]:
    ctl = make_control(RMODEL + ASPGARPFILES, MAX_STATES)

    ctl.add("base", [], "#show obs/1.")
    ctl.add("base", [], ":- violation.")
    ctl.ground([("base", [])])

    states: list[State] = []

    with ctl.solve(yield_=True) as handle:
        for model in handle:
            state: State = {}

            for atom in model.symbols(shown=True):
                if atom.name != "obs" or len(atom.arguments) != 1:
                    continue

                args = atom.arguments[0].arguments

                if len(args) == 3:
                    varname, value, direction = map(str, args)
                    state[varname] = (value, direction)

                elif len(args) == 2:
                    varname, value = map(str, args)
                    state[varname] = [value]

            states.append(state)

    return states


def states_to_description(states: list[State]) -> list[str]:
    def part(varname: Any, item: Any) -> str:
        if isinstance(item, tuple) and len(item) == 2:
            value, direction = item
            return f"obsT(holds(SID,{varname},{value},{direction}))"

        return f"obsT(holds_relation(SID,{varname},{item[0]}))"

    return [
        f"state(SID,{i}) :- "
        + ", ".join(part(varname, item) for varname, item in state.items())
        + "."
        for i, state in enumerate(states)
    ]


def asp_term(x: Any) -> str:
    if isinstance(x, list):
        return asp_term(x[0]) if len(x) == 1 else "(" + ",".join(map(asp_term, x)) + ")"

    if isinstance(x, tuple):
        return "(" + ",".join(map(asp_term, x)) + ")"

    return str(x)


def states_to_labels(states: list[State]) -> list[str]:
    def part(varname: Any, item: Any) -> str:
        var = asp_term(varname)

        if isinstance(item, tuple) and len(item) == 2:
            value, direction = item
            return f"{var}=({asp_term(value)},{asp_term(direction)})"

        if isinstance(item, list) and len(item) == 1:
            return f"{var}=({asp_term(item[0])})"

        return f"{var}=({asp_term(item)})"

    return [
        f'statelabel({i},"{",\\n ".join(part(varname, item) for varname, item in state.items())}").'
        for i, state in enumerate(states)
    ]


def states_to_dataframe(states: list[State]) -> pd.DataFrame:
    varnames = sorted({varname for state in states for varname in state})

    return pd.DataFrame(
        {
            "StateID": i,
            **{
                varname: (
                    f"{value[0]},{value[1]}"
                    if len(value := state.get(varname, ("N/A", "N/A"))) == 2
                    else value[0]
                )
                for varname in varnames
            },
        }
        for i, state in enumerate(states)
    )


def transitions_from_reference_model(
    RMODEL: list[str],
    ASPGARPFILES: list[str],
    state_descriptions: list[str],
) -> list[tuple[int, int]]:
    ctl = make_control(RMODEL + ASPGARPFILES, 0)

    ctl.add("base", [], "\n".join(state_descriptions))
    ctl.add("base", [], ":- violation.")
    ctl.add("base", [], "#show state/2.")
    ctl.ground([("base", [])])

    transitions: list[tuple[int, int]] = []

    with ctl.solve(yield_=True) as handle:
        for i, model in enumerate(handle):

            pair: dict[int, int] = {}

            for atom in model.symbols(shown=True):
                if atom.name != "state" or len(atom.arguments) != 2:
                    continue

                sid, state_id = atom.arguments

                if (
                    sid.type == clingo.SymbolType.Number
                    and state_id.type == clingo.SymbolType.Number
                ):
                    pair[sid.number] = state_id.number

            if 0 in pair and 1 in pair:
                transitions.append((pair[0], pair[1]))
            else:
                print("Warning: Could not find both from_state and to_state in the model.")

    return transitions

def state_2_clingo_assumptions(state: dict[str, tuple[str, str]]):
    assumptions = []
    for varname, (value, direction) in state.items():
        assumptions.append((clingo.parse_term(f"holds({varname}, {value}, {direction})"), True))
    return assumptions

def states_2_clingo_assumptions(states):
    assumptions=[]
    for sid,state in enumerate(states):
        for varname, (value, direction) in state.items():
            assumptions.append((clingo.parse_term(f"holds({sid},{varname}, {value}, {direction})"), True))
    return assumptions


# Setup

ASP encodings ...
- ... for Qualitative Simulation
- ... Qualitative Constraint Network
- ... Garp3 (Causal Dependencies / Correspondences / Calculi)



In [ ]:
INSTANCE = "tree"

ASPGARP_FILES = [
    f"{project_root}/prototype/single/qsim.lp",
    f"{project_root}/prototype/single/cd.lp",
    f"{project_root}/prototype/adapter.lp",
    f"{project_root}/prototype/db.lp",
    f"{project_root}/prototype/preds.lp",
    f"{project_root}/prototype/single/garp_dynamics.lp",
    f"{project_root}/prototype/qcn/encoding.lp",
    f"{project_root}/prototype/qcn/calculi/point.lp",
    f"{project_root}/prototype/single/calculi.lp",
    f"{project_root}/prototype/single/qcn_dynamics.lp",
]

MULTI = [
    f"{project_root}/prototype/single/multi.lp",
]

SINGLE = [
    f"{project_root}/prototype/single/single.lp",
]
REFERENCE_MODEL = [
    f"{project_root}/models/{INSTANCE}/db.lp",
    f"{project_root}/models/{INSTANCE}/ref.lp"
]

BASIC_MODEL = [
    f"{project_root}/models/{INSTANCE}/db.lp"
]

META_FILES = [
    f"{project_root}/prototype/meta.lp",
    f"{project_root}/prototype/well_founded.lp",
]

OUTPUT = f"{project_root}/out/{INSTANCE}"

MAX_STATES = "0"

#create output directory if it doesn't exist
os.makedirs(OUTPUT, exist_ok=True)

## Reference Model (Ground Truth)

In [ ]:
from IPython.display import SVG, TextDisplayObject, Markdown
MODEL_DB = REFERENCE_MODEL[0]
MODEL_REF = REFERENCE_MODEL[1]
!clingo 1 --project --warn=none --outf=2 {MODEL_DB} {MODEL_REF} | clingraph --out=render --format=svg --viz-encoding={project_root}/prototype/viz/causal.lp --dir={OUTPUT}/visuals --default-graph=causal
SVG(filename=f"{OUTPUT}/visuals/0/causal.svg")

### Generate Full State Graph

In [ ]:

states = states_from_reference_model(REFERENCE_MODEL, ASPGARP_FILES + SINGLE)

df = states_to_dataframe(states)
df.to_csv(f"{OUTPUT}/states.csv", index=False)
display(df)

state_descriptions = states_to_description(states)
state_labels = states_to_labels(states)
with open(f"{OUTPUT}/states.lp", "w") as f:
    f.write("\n".join(state_descriptions))

print(f"-> Saved states to {OUTPUT}/states.csv")

### Find transitions between States

In [ ]:

transitions = transitions_from_reference_model(REFERENCE_MODEL, ASPGARP_FILES + SINGLE, state_descriptions)

transition_descriptions = [f"edge(({from_state},{to_state}))." for from_state, to_state in transitions]

with open(f"{OUTPUT}/transitions.lp", "w") as f:
    f.write("\n".join(transition_descriptions))
    f.write("\n".join(state_labels))

!clingo --project --warn=none --outf=2 {OUTPUT}/transitions.lp  | clingraph --out=render --format=pdf --viz-encoding="{project_root}/viz.lp" --dir={OUTPUT}/visuals --default-graph=states
SVG(filename=f"{OUTPUT}/visuals/0/states.svg")


# Abduction


In [ ]:
# def justify_state(state: dict[str, tuple[str, str]], reject = False, max_size=5):
#     assumptions = state_2_clingo_assumptions(state)
#     assumptions.append((clingo.parse_term("violation"), reject)) 
#     for i in range(0,max_size):
#         ctl = clingo.Control(["0", "--const", f"r={i}", "--project", "--warn=none"])
#         for path in ASPGARP_FILES+ SINGLE + META_FILES + BASIC_MODEL:
#             ctl.load(path)
#         ctl.add("base", [], "#show rule/3.")
#         ctl.ground([("base", [])])
#         success = False
#         with ctl.solve(yield_=True,assumptions=assumptions) as handle:
#             for model in handle:
#                 yield model.symbols(shown=True)
#                 success = True
#             if success:
#                 return
#     raise Exception(f"Could not justify state with r up to {max_size-1}.")


In [ ]:
def justify_states(states, reject = False, max_size=10,base_model=""):
    
    assumptions = []
    assumptions = states_2_clingo_assumptions(states)
    
    for i in range(0,max_size):
        #print(f"Trying to justify states with r={i}...")
        ctl = clingo.Control(["0", "--const", f"r={i}", "--project", "--warn=none"])
        for path in ASPGARP_FILES+ MULTI + META_FILES + BASIC_MODEL:
            ctl.load(path)
        ctl.add("base", [], base_model)
        ctl.add("base", [], "#show rule/3.")
        for sid,state in enumerate(states):
            ctl.add("base", [], f"sim({sid}).")
            assumptions.append((clingo.parse_term(f"sviolation({sid})"), reject)) 
        #print(f"Grounding with {len(assumptions)} assumptions...")
        ctl.ground([("base", [])])
        #print(f"Solving with {len(assumptions)} assumptions...")
        success = False
        with ctl.solve(yield_=True,assumptions=assumptions) as handle:
            for model in handle:
                #print(f"Found justification for states with r={i}.")
                yield model.symbols(shown=True)
                success = True
            if success:
                return
    raise Exception(f"Could not justify state with r up to {max_size-1}.")


In [ ]:
base_model = """
precond(all, true).

rule(r1,all,influence(-1, flow, lamount)).

rule(r2,all,influence(1, flow, ramount)).

rule(r3,all,proportionality(1, lamount, llevel)).

rule(r4,all,proportionality(1, ramount, rlevel)).

rule(r5,all,correspondence(1,lamount, llevel)).
rule(r6,all,correspondence(1,ramount, rlevel)).

rule(r7,all,proportionality(1, llevel, lpressure)).
rule(r8,all,proportionality(1, rlevel, rpressure)).

rule(r9,all,correspondence(1,lamount, lpressure)).
rule(r10,all,correspondence(1,ramount, rpressure)).

rule(r11,all,proportionality(1, lpressure, flow)).
rule(r12,all,proportionality(-1, rpressure, flow)).

rule(r13,all,landmark_relation(lheight, land(point), llevel, land(max), eq)).
rule(r14,all,landmark_relation(rheight, land(point), rlevel, land(max), eq)).

rule(r15,all,landmark_relation(rheight, land(point), lheight, land(point), eq)).

rule(r16,all,minus(lpressure, rpressure, flow)).

rule(r17,all,algebraic(llevel, eq, lpressure)).
rule(r18,all,algebraic(rlevel, eq, rpressure)).

"""


In [ ]:
for model in justify_states(states):
    print(model)

